<a href="https://colab.research.google.com/github/boss-defender/FineTune/blob/main/SmartFineTuner.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==============================================================================
# STEP 1: GPU AVAILABILITY CHECK ⚡
# ==============================================================================
import torch

if not torch.cuda.is_available():
    raise SystemError("❌ NO GPU FOUND! Go to Runtime -> Change runtime type -> Select T4 GPU!")
print(f"✅ GPU DETECTED: {torch.cuda.get_device_name(0)}! Hardware verified.")

# ==============================================================================
# STEP 2: GOOGLE DRIVE MOUNT 💾
# ==============================================================================
from google.colab import drive
import os
import json
import hashlib
import re

drive.mount('/content/drive')
print("✅ Google Drive connected!")

# ==============================================================================
# STEP 3: AUTO-INSTALL DEPENDENCIES 📦
# ==============================================================================
print("\n🔄 Installing latest Unsloth & export tools...")
!pip install --quiet --upgrade "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --quiet --upgrade --no-deps trl peft accelerate bitsandbytes datasets transformers xformers huggingface_hub
print("✅ Packages updated!")

# ==============================================================================
# STEP 4: CONFIGURATION & CONTROL PANEL 🎯
# ==============================================================================
MODEL_NAME = "unsloth/Llama-3.2-3B-Instruct"    #@param {type:"string"}
MODEL_REVISION = "main"                         #param {type:"string"}
DATASET_NAME = "philschmid/dolly-15k-oai-style"  #@param {type:"string"}
DATASET_REVISION = "main"                       #param {type:"string"}
DATASET_SPLIT = "train"                        #param {type:"string"}

# Hyperparameters
MAX_SEQ_LENGTH = 2048                           #param {type:"integer"}
LEARNING_RATE = 2e-4                            #param {type:"number"}
LORA_R = 16                                     #param {type:"integer"}
LORA_ALPHA = 16                                 #param {type:"integer"}
LORA_DROPOUT = 0.0                              #param {type:"number"}
BATCH_SIZE = 2                                  #param {type:"integer"}
GRAD_ACCUM_STEPS = 4                            #param {type:"integer"}
MAX_STEPS = 200                                 #param {type:"integer"}
WARMUP_STEPS = 5                                #param {type:"integer"}
WEIGHT_DECAY = 0.01                             #param {type:"number"}
LR_SCHEDULER_TYPE = "linear"                    #param {type:"string"}
OPTIMIZER = "adamw_8bit"                        #param {type:"string"}
SEED = 3407                                     #param {type:"integer"}
SAVE_CHECKPOINT_STEPS = 25                      #param {type:"integer"}

TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

# ==============================================================================
# STEP 5: CONFIG HASHING & DIRECTORY LOCK 🔒
# ==============================================================================
# 1. Complete parameter dictionary capturing exact hyperparameter state
RUN_CONFIG = {
    "model_name": MODEL_NAME,
    "model_revision": MODEL_REVISION,
    "dataset_name": DATASET_NAME,
    "dataset_revision": DATASET_REVISION,
    "dataset_split": DATASET_SPLIT,
    "max_seq_length": MAX_SEQ_LENGTH,
    "learning_rate": LEARNING_RATE,
    "lora_r": LORA_R,
    "lora_alpha": LORA_ALPHA,
    "lora_dropout": LORA_DROPOUT,
    "target_modules": TARGET_MODULES,
    "batch_size": BATCH_SIZE,
    "grad_accum_steps": GRAD_ACCUM_STEPS,
    "max_steps": MAX_STEPS,
    "warmup_steps": WARMUP_STEPS,
    "weight_decay": WEIGHT_DECAY,
    "lr_scheduler_type": LR_SCHEDULER_TYPE,
    "optimizer": OPTIMIZER,
    "seed": SEED,
}

# 2. Compute a SHA-256 hash of ALL parameters
config_json_str = json.dumps(RUN_CONFIG, sort_keys=True)
config_hash = hashlib.sha256(config_json_str.encode('utf-8')).hexdigest()[:8]

# 3. Unique folder paths
clean_model = re.sub(r'[^a-zA-Z0-9_\-]', '_', MODEL_NAME.split('/')[-1])
clean_dataset = re.sub(r'[^a-zA-Z0-9_\-]', '_', DATASET_NAME.split('/')[-1])

RUN_FOLDER_NAME = f"{clean_model}__{clean_dataset}__{config_hash}"

DRIVE_CHECKPOINT_DIR = f"/content/drive/MyDrive/unsloth_checkpoints/{RUN_FOLDER_NAME}"
LOCAL_FINAL_SAVE_DIR = f"/content/merged_16bit_model"

print(f"\n📂 UNIQUE RUN DIRECTORY: {RUN_FOLDER_NAME}")
print(f"🔒 SHA-256 Config Hash: {config_hash}")

# 4. Verify or create run_config.json in Google Drive
os.makedirs(DRIVE_CHECKPOINT_DIR, exist_ok=True)
CONFIG_FILE_PATH = os.path.join(DRIVE_CHECKPOINT_DIR, "run_config.json")

if os.path.exists(CONFIG_FILE_PATH):
    with open(CONFIG_FILE_PATH, "r") as f:
        existing_config = json.load(f)
    if existing_config == RUN_CONFIG:
        print("✅ Config Verification PASSED: All parameters match saved Drive run!")
    else:
        raise ValueError("❌ CONFIG MISMATCH DETECTED! Saved parameters do not match your current settings.")
else:
    with open(CONFIG_FILE_PATH, "w") as f:
        json.dump(RUN_CONFIG, f, indent=4)
    print("📝 Saved comprehensive run_config.json to Google Drive directory.")

# ==============================================================================
# STEP 6: LOAD MODEL & TOKENIZER 🤖
# ==============================================================================
from unsloth import FastLanguageModel

print(f"\n📥 Loading Base Model: {MODEL_NAME} (Revision: {MODEL_REVISION})...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_NAME,
    revision = MODEL_REVISION,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype = None,
    load_in_4bit = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = LORA_R,
    target_modules = TARGET_MODULES,
    lora_alpha = LORA_ALPHA,
    lora_dropout = LORA_DROPOUT,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = SEED,
)

# ==============================================================================
# STEP 7: LOAD & FORMAT DATASET 📑
# ==============================================================================
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template

print(f"\n📥 Loading Dataset: {DATASET_NAME} (Revision: {DATASET_REVISION})...")
dataset = load_dataset(DATASET_NAME, split = DATASET_SPLIT, revision = DATASET_REVISION)

dataset_text_field = "text"
if "messages" in dataset.column_names:
    tokenizer = get_chat_template(tokenizer, chat_template = "llama-3")
    def format_chat(examples):
        texts = [tokenizer.apply_chat_template(msg, tokenize=False, add_generation_prompt=False) for msg in examples["messages"]]
        return {"text": texts}
    dataset = dataset.map(format_chat, batched=True)

# ==============================================================================
# STEP 8: CONFIGURE TRAINER & RESUME LOGIC 🔄
# ==============================================================================
from trl import SFTTrainer
from transformers import TrainingArguments
from transformers.trainer_utils import get_last_checkpoint

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = dataset_text_field,
    max_seq_length = MAX_SEQ_LENGTH,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = BATCH_SIZE,
        gradient_accumulation_steps = GRAD_ACCUM_STEPS,
        warmup_steps = WARMUP_STEPS,
        max_steps = MAX_STEPS,
        learning_rate = LEARNING_RATE,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = OPTIMIZER,
        weight_decay = WEIGHT_DECAY,
        lr_scheduler_type = LR_SCHEDULER_TYPE,
        seed = SEED,
        output_dir = DRIVE_CHECKPOINT_DIR,
        save_strategy = "steps",
        save_steps = SAVE_CHECKPOINT_STEPS,
        save_total_limit = 2,
    ),
)

last_checkpoint = get_last_checkpoint(DRIVE_CHECKPOINT_DIR) if os.path.exists(DRIVE_CHECKPOINT_DIR) else None

if last_checkpoint is not None:
    print(f"\n🔍 FOUND CHECKPOINT: {last_checkpoint}")
    print("⚡ Resuming training seamlessly...\n")
    trainer.train(resume_from_checkpoint = last_checkpoint)
else:
    print("\n🚀 Starting fresh fine-tuning run...\n")
    trainer.train()

# ==============================================================================
# STEP 9: EXPORT SINGLE MERGED STANDALONE MODEL 📦
# ==============================================================================
print(f"\n🎉 Training Complete! Merging weights and saving ONCE to '{LOCAL_FINAL_SAVE_DIR}'...")

model.save_pretrained_merged(
    LOCAL_FINAL_SAVE_DIR,
    tokenizer,
    save_method = "merged_16bit",
)

print(f"✅ Merged 16-bit base model saved successfully to '{LOCAL_FINAL_SAVE_DIR}'!")

In [ ]:
# ==============================================================================
# LIGHTWEIGHT HF UPLOADER (NO UNSLOTH NEEDED! 🚀)
# ==============================================================================
!pip install --quiet huggingface_hub

from huggingface_hub import HfApi
import os

# 1. CONFIGURATION PANEL 🎛️
HF_WRITE_TOKEN = "hf_xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx"  #@param {type:"string"}
HF_REPO_NAME = "your-username/my-finetuned-model"        #@param {type:"string"}
FOLDER_PATH = "/content/merged_16bit_model"             #@param {type:"string"}
REPO_VISIBILITY = "public"                             #@param ["private", "public"]

# 2. VERIFY FOLDER EXISTS
if not os.path.exists(FOLDER_PATH):
    raise FileNotFoundError(f"❌ Cannot find folder at: {FOLDER_PATH}")

# 3. INITIALIZE HF API & CREATE REPO
api = HfApi(token=HF_WRITE_TOKEN)

print(f"📁 Preparing Hugging Face repository '{HF_REPO_NAME}' ({REPO_VISIBILITY.upper()})...")
api.create_repo(
    repo_id=HF_REPO_NAME,
    private=(REPO_VISIBILITY == "private"),
    exist_ok=True,
    repo_type="model"
)

# 4. PUSH FOLDER TO HUGGING FACE
print(f"🚀 Uploading all files from '{FOLDER_PATH}'...")
api.upload_folder(
    folder_path=FOLDER_PATH,
    repo_id=HF_REPO_NAME,
    repo_type="model",
)

print(f"\n🎉 BOOM! Your model is live at: https://huggingface.co/{HF_REPO_NAME}")